In [0]:
"""
Data Quality - Aggregate Checks Task
Consolidates multiple checks per dimension and calculates overall check
"""
from pyspark.sql import functions as F
import re
import json

# Get parameters from notebook widgets
dbutils.widgets.text("project_id", "")
dbutils.widgets.text("project_name", "")
dbutils.widgets.text("job_name", "")
dbutils.widgets.text("catalog", "dbr_uat01_gdm")
dbutils.widgets.text("schema", "db_gdm_data_quality_app")
dbutils.widgets.text("results_schema", "db_gdm_dq_results")
dbutils.widgets.text("enabled_dimensions", "[]")
dbutils.widgets.text("columns", "[]")

PROJECT_ID = dbutils.widgets.get("project_id")
PROJECT_NAME = dbutils.widgets.get("project_name")
JOB_NAME = dbutils.widgets.get("job_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
RESULTS_SCHEMA = dbutils.widgets.get("results_schema")
ENABLED_DIMENSIONS = json.loads(dbutils.widgets.get("enabled_dimensions"))
COLUMNS_TO_CHECK = json.loads(dbutils.widgets.get("columns"))

# Sanitize names for table naming
def sanitize_name(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name).lower()

project_name_clean = sanitize_name(PROJECT_NAME) if PROJECT_NAME else f"project_{PROJECT_ID}"
job_name_clean = sanitize_name(JOB_NAME) if JOB_NAME else f"job_{PROJECT_ID}"

print(f"=== Data Quality - Aggregate Checks ===")
print(f"Project: {PROJECT_NAME} (ID: {PROJECT_ID})")
print(f"Job: {JOB_NAME}")
print(f"Project name clean: {project_name_clean}")
print(f"Job name clean: {job_name_clean}")
print(f"Results Schema: {RESULTS_SCHEMA}")
print(f"Enabled dimensions: {ENABLED_DIMENSIONS}")
print(f"Columns: {COLUMNS_TO_CHECK}")

# Start with row_id from input
input_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_input"
print(f"\nReading input from: {input_table}")
df_agg = spark.table(input_table).select("row_id")

# Process each dimension
for dimension in ENABLED_DIMENSIONS:
    dimension_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_{dimension}"
    
    print(f"\nLooking for dimension table: {dimension_table}")
    
    try:
        df_dimension = spark.table(dimension_table)
        print(f"✓ Found {dimension} table with {df_dimension.count()} rows")
        
        # Get all check columns for this dimension
        dimension_check_cols = [col for col in df_dimension.columns if col.endswith(f"_{dimension}_check")]
        
        print(f"  Check columns found: {dimension_check_cols}")
        
        # Group by column base name (handle multiple checks per column)
        column_checks = {}
        for check_col in dimension_check_cols:
            # Extract base column name (remove _NNN_dimension_check or _dimension_check)
            pattern = rf"(.+?)(_\d{{3}})?_{dimension}_check$"
            match = re.match(pattern, check_col)
            if match:
                base_col = match.group(1)
                if base_col not in column_checks:
                    column_checks[base_col] = []
                column_checks[base_col].append(check_col)
        
        # Merge multiple checks per column
        for base_col, check_cols in column_checks.items():
            consolidated_col = f"{base_col}_{dimension}_check"
            
            if len(check_cols) == 1:
                # Single check, just rename
                df_agg = df_agg.join(
                    df_dimension.select("row_id", check_cols[0]).withColumnRenamed(check_cols[0], consolidated_col),
                    "row_id",
                    "left"
                )
                print(f"  {base_col}: Single check → {consolidated_col}")
            else:
                # Multiple checks, apply aggregation logic:
                # - 1 if ALL checks are 1
                # - 0 if ANY check is 0
                # - NULL if all are NULL
                print(f"  {base_col}: Merging {len(check_cols)} checks → {consolidated_col}")
                
                check_array = F.array(*[F.col(c) for c in check_cols])
                merged_check = F.when(
                    F.array_contains(check_array, 0), 0  # ANY check failed
                ).when(
                    F.forall(check_array, lambda x: x == 1), 1  # ALL checks passed
                ).otherwise(None)  # All NULL
                
                df_temp = df_dimension.select("row_id", *check_cols).withColumn(
                    consolidated_col,
                    merged_check
                )
                
                df_agg = df_agg.join(
                    df_temp.select("row_id", consolidated_col),
                    "row_id",
                    "left"
                )
        
    except Exception as e:
        print(f"✗ Could not process {dimension} table: {str(e)}")

if "accuracy" not in ENABLED_DIMENSIONS:
    # Process accuracy dimension (special handling for multiple rules per column)
    print("\nProcessing accuracy dimension:")
    accuracy_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_accuracy"
    try:
        df_accuracy = spark.table(accuracy_table)
        print(f"✓ Found accuracy table with {df_accuracy.count()} rows")
        
        # Get all check columns for accuracy (pattern: {col}_{idx}_accuracy_check)
        accuracy_check_cols = [col for col in df_accuracy.columns if col.endswith("_accuracy_check")]
        print(f"  Check columns found: {accuracy_check_cols}")
        
        # Group by column base name (handle multiple rules per column)
        column_checks = {}
        for check_col in accuracy_check_cols:
            # Extract base column name (remove _NNN_accuracy_check)
            pattern = rf"(.+?)_\d{{3}}_accuracy_check$"
            match = re.match(pattern, check_col)
            if match:
                base_col = match.group(1)
                if base_col not in column_checks:
                    column_checks[base_col] = []
                column_checks[base_col].append(check_col)
        
        # Merge multiple checks (rules) per column
        for base_col, check_cols in column_checks.items():
            consolidated_col = f"{base_col}_accuracy_check"
            
            if len(check_cols) == 1:
                # Single rule, just rename
                df_agg = df_agg.join(
                    df_accuracy.select("row_id", check_cols[0]).withColumnRenamed(check_cols[0], consolidated_col),
                    "row_id",
                    "left"
                )
                print(f"  {base_col}: Single rule → {consolidated_col}")
            else:
                # Multiple rules, apply aggregation logic:
                # - 1 if ALL rules pass
                # - 0 if ANY rule fails
                # - NULL if all are NULL
                print(f"  {base_col}: Merging {len(check_cols)} rules → {consolidated_col}")
                
                check_array = F.array(*[F.col(c) for c in check_cols])
                merged_check = F.when(
                    F.array_contains(check_array, 0), 0  # ANY rule failed
                ).when(
                    F.forall(check_array, lambda x: x == 1), 1  # ALL rules passed
                ).otherwise(None)  # All NULL
                
                df_temp = df_accuracy.select("row_id", *check_cols).withColumn(
                    consolidated_col,
                    merged_check
                )
                
                df_agg = df_agg.join(
                    df_temp.select("row_id", consolidated_col),
                    "row_id",
                    "left"
                )
    except Exception as e:
        print(f"✗ Could not process accuracy table: {str(e)}")

# Calculate overall check per column
# overall_check = 1 if ALL dimension checks are 1, 0 if ANY is 0, NULL otherwise
print("\nCalculating overall checks per column:")
for col_name in COLUMNS_TO_CHECK:
    overall_col = f"{col_name}_overall_check"
    
    # Get all dimension checks for this column
    dimension_checks = [f"{col_name}_{dim}_check" for dim in ENABLED_DIMENSIONS 
                       if f"{col_name}_{dim}_check" in df_agg.columns]
    
    if dimension_checks:
        print(f"  {col_name}: {len(dimension_checks)} dimension(s)")
        
        check_array = F.array(*[F.col(c) for c in dimension_checks])
        overall_check = F.when(
            F.array_contains(check_array, 0), 0  # ANY dimension failed
        ).when(
            F.forall(check_array, lambda x: x == 1), 1  # ALL dimensions passed
        ).otherwise(None)  # All NULL
        
        df_agg = df_agg.withColumn(overall_col, overall_check)

# Add execution timestamp
df_agg = df_agg.withColumn("execution_timestamp", F.current_timestamp())

# Output table name
output_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_checks_aggregated"

print(f"\nWriting to: {output_table}")

# Write to Unity Catalog
df_agg.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"✅ Successfully wrote {df_agg.count()} rows to {output_table}")

# Display sample
print("\nSample data (first 5 rows):")
df_agg.show(5, truncate=False)